# Mini Project 9 - Part 2: Traditional ML Baseline
## TF-IDF + Logistic Regression

**Goal:** Build a traditional ML baseline to compare against the transformer.

**Approach:** TF-IDF vectorization + Logistic Regression with hyperparameter tuning

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, 
    classification_report, 
    confusion_matrix,
    f1_score
)
from sklearn.model_selection import GridSearchCV
import warnings
warnings.filterwarnings('ignore')

SEED = 42
sns.set_style('whitegrid')

## 1. Load Processed Data

In [ ]:
# Load splits from exploration notebook
train = pd.read_csv('data/train.csv')
val = pd.read_csv('data/val.csv')
test = pd.read_csv('data/test.csv')

print(f"Train: {len(train)} samples")
print(f"Val:   {len(val)} samples")
print(f"Test:  {len(test)} samples")

# Use aggressively cleaned text for TF-IDF
X_train = train['text_clean_aggressive'].values
y_train = train['class'].values

X_val = val['text_clean_aggressive'].values
y_val = val['class'].values

X_test = test['text_clean_aggressive'].values
y_test = test['class'].values

## 2. TF-IDF Vectorization

We'll experiment with:
- **max_features**: Vocabulary size
- **ngram_range**: Unigrams vs bigrams
- **min_df**: Minimum document frequency

In [ ]:
# Create TF-IDF vectorizer
# We'll tune these parameters
vectorizer = TfidfVectorizer(
    max_features=5000,      # Top 5000 most frequent words
    ngram_range=(1, 2),     # Unigrams and bigrams
    min_df=2,               # Word must appear in at least 2 documents
    max_df=0.9,             # Ignore words in >90% of documents
    strip_accents='unicode',
    stop_words='english'    # Remove common English stop words
)

# Fit on training data and transform all splits
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)
X_test_tfidf = vectorizer.transform(X_test)

print(f"TF-IDF matrix shape (train): {X_train_tfidf.shape}")
print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")
print(f"\nTop 20 features:")
feature_names = vectorizer.get_feature_names_out()
print(feature_names[:20])

## 3. Baseline Model: Logistic Regression

Using `class_weight='balanced'` to handle class imbalance.

In [ ]:
# Train baseline model
baseline_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',  # Handle imbalance
    random_state=SEED,
    n_jobs=-1
)

print("Training baseline model...")
baseline_model.fit(X_train_tfidf, y_train)
print("✅ Training complete!")

# Predictions
y_val_pred = baseline_model.predict(X_val_tfidf)
y_test_pred = baseline_model.predict(X_test_tfidf)

# Accuracy
val_acc = accuracy_score(y_val, y_val_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print(f"\nValidation Accuracy: {val_acc:.4f}")
print(f"Test Accuracy:       {test_acc:.4f}")

## 4. Hyperparameter Tuning

Tune `C` (regularization strength) using GridSearchCV.

In [ ]:
# Parameter grid
param_grid = {
    'C': [0.1, 1.0, 10.0, 100.0],  # Regularization strength
}

# Grid search with F1-macro scoring (better for imbalanced data)
grid_search = GridSearchCV(
    LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED, n_jobs=-1),
    param_grid,
    cv=3,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

print("Running grid search...")
grid_search.fit(X_train_tfidf, y_train)

print(f"\n✅ Best parameters: {grid_search.best_params_}")
print(f"Best cross-validation F1 (macro): {grid_search.best_score_:.4f}")

# Use best model
best_model = grid_search.best_estimator_

## 5. Evaluation on Test Set

In [ ]:
# Predictions with best model
y_test_pred_best = best_model.predict(X_test_tfidf)
y_test_proba = best_model.predict_proba(X_test_tfidf)

# Metrics
test_acc_best = accuracy_score(y_test, y_test_pred_best)
test_f1_macro = f1_score(y_test, y_test_pred_best, average='macro')
test_f1_weighted = f1_score(y_test, y_test_pred_best, average='weighted')

print("="*70)
print("TF-IDF BASELINE - FINAL TEST RESULTS")
print("="*70)
print(f"Accuracy:        {test_acc_best:.4f}")
print(f"F1 (macro):      {test_f1_macro:.4f}")
print(f"F1 (weighted):   {test_f1_weighted:.4f}")
print("="*70)

## 6. Per-Class Performance

In [ ]:
# Classification report
class_names = ['Hate speech', 'Offensive', 'Neither']
print("\nClassification Report:")
print(classification_report(y_test, y_test_pred_best, target_names=class_names, digits=4))

## 7. Confusion Matrix

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(y_test, y_test_pred_best)

# Normalize for percentages
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set_ylabel('True Label', fontsize=12)
axes[0].set_xlabel('Predicted Label', fontsize=12)
axes[0].set_title('Confusion Matrix (Counts)', fontsize=14, fontweight='bold')

# Percentages
sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=axes[1])
axes[1].set_ylabel('True Label', fontsize=12)
axes[1].set_xlabel('Predicted Label', fontsize=12)
axes[1].set_title('Confusion Matrix (Percentages)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 8. Error Analysis: Hardest Examples

In [ ]:
# Find misclassified examples
test_df = test.copy()
test_df['predicted'] = y_test_pred_best
test_df['confidence'] = y_test_proba.max(axis=1)
test_df['correct'] = (test_df['class'] == test_df['predicted'])

# Misclassified examples
errors = test_df[~test_df['correct']]

print(f"Total errors: {len(errors)} / {len(test_df)} ({len(errors)/len(test_df)*100:.2f}%)")
print("\nError breakdown by true class:")
for cls in [0, 1, 2]:
    label = {0: 'Hate speech', 1: 'Offensive', 2: 'Neither'}[cls]
    cls_errors = len(errors[errors['class'] == cls])
    cls_total = len(test_df[test_df['class'] == cls])
    print(f"  {label:15s}: {cls_errors:3d} / {cls_total:4d} ({cls_errors/cls_total*100:5.2f}% error rate)")

In [ ]:
# Show 10 hardest misclassified examples (lowest confidence)
hardest_errors = errors.nsmallest(10, 'confidence')

print("\n" + "="*80)
print("10 HARDEST MISCLASSIFICATIONS (Lowest Confidence)")
print("="*80)

for idx, (i, row) in enumerate(hardest_errors.iterrows(), 1):
    true_label = {0: 'Hate', 1: 'Offensive', 2: 'Neither'}[row['class']]
    pred_label = {0: 'Hate', 1: 'Offensive', 2: 'Neither'}[row['predicted']]
    
    print(f"\nExample {idx}:")
    print(f"  Text: {row['text_clean']}")
    print(f"  True: {true_label} | Predicted: {pred_label} | Confidence: {row['confidence']:.3f}")

## 9. Save Baseline Results

In [ ]:
# Save results for comparison with transformer
baseline_results = {
    'model': 'TF-IDF + Logistic Regression',
    'test_accuracy': test_acc_best,
    'test_f1_macro': test_f1_macro,
    'test_f1_weighted': test_f1_weighted,
    'best_params': grid_search.best_params_,
    'confusion_matrix': cm.tolist()
}

import json
with open('data/baseline_results.json', 'w') as f:
    json.dump(baseline_results, f, indent=2)

print("✅ Baseline results saved to ../data/baseline_results.json")

---

## Summary: TF-IDF Baseline

**Final Results:**
- Test Accuracy: [fill in]
- F1 (macro): [fill in]
- F1 (weighted): [fill in]

**Observations:**
1. **Best performing class**: [analyze confusion matrix]
2. **Worst performing class**: [likely hate speech due to imbalance]
3. **Common errors**: [hate vs offensive confusion]

**Next:** Notebook 3 - Can DistilBERT beat this baseline?